# 西塔访客分析

## 配置更新
把接下来用到的配置写入到配置文件当中
```yaml
# configs/settings.yaml —— 业务参数的唯一来源（AGENTS.md 第 8 条）
# 改参数改这里，不改代码；代码里不写 500 米、30 分钟这类业务数字。
# 一律 UTF-8 保存（Windows 记事本另存为时选 UTF-8）。
version: "1.0"

hotspot:                 # 热点：D04 定的，D06 起从这里读
  name: 西塔
  top_n: 5               # 连接用户数前几的站点参与加权，定热点中心
  radius_m: 500          # 热点范围：离中心多少米以内算“在西塔”

stay:                    # 停留识别（D06）
  radius_m: 500          # 按距离切段的半径：吸收乒乓切换；300–1000 米结果差别不大
  min_minutes: 30        # 停留的最短时长：结果对它敏感，必须写进口径

# ── D07–D08 分析口径（D07 开工时追加；上面的内容一个字不改）──────────────
visitor:
  evening_hour: 17        # 傍晚访客：西塔停留的结束时刻不早于 17:00
  late_hour: 23           # 删失：23:00 以后仍在西塔，停留时长只是下限
  near_km: 1.5            # 周边居民：过夜地离西塔不到 1.5 公里
  
night:
  start_hour: 0           # 过夜地：与 0:00–6:00 重叠最长的停留
  end_hour: 6
  min_overlap_minutes: 60 # 重叠不到 60 分钟的不算过夜地
  
hourly:
  min_cover_minutes: 30   # 每人每小时的位置：覆盖该小时 ≥ 30 分钟的停留优先
privacy:
  min_cell_users: 5       # 对外交付：少于 5 人的格子不展示
```

In [ ]:
# ── 第一格：环境准备（每本 notebook 开头都是这一段）──
%load_ext autoreload
%autoreload 2
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, db, geo, pipeline, viz
from signal_insight.viz import (MAIN, MAIN_LIGHT, MAIN_DARK,
MAIN_MID, GRAY, GRAY_DARK, INK, MUTED, FAINT, AREA_COLOR)

viz.apply_theme()                   # 字体、配色、坐标轴样式
con = sqlite3.connect(config.DB_PATH)
DAY = config.ANALYSIS_DAY
DAY_START = pd.Timestamp(DAY)        # 2018-10-03 00:00：后面所有“几点”都从这个时刻算起
NB = "D07_西塔访客"
SOURCE = f"{config.SOURCE_NOTE}；制图：{config.AUTHOR}"
pd.set_option("display.max_columns", 30)


def mask_id(imsi):
    """展示用户时只显示 IMSI 尾号。D06 里也有一份：notebook 是记录，
    各带一份没关系，不回去改 D06。"""
    return "尾号" + str(imsi)[-4:]

In [ ]:
log = pipeline.run_day()

In [ ]:
log

In [ ]:
log.tail()

In [ ]:
# 读取今天要用的数据
P = config.load_settings()
hot = pd.read_sql("SELECT * FROM dim_hotspot", con).iloc[0]
R = P["hotspot"]['radius_m']

# 获取区县，商圈的边界
DISTRICTS = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), 'district')
AOIS = geo.rings_from_points(pd.read_sql("SELECT * FROM ods_aoi_v2", con),
                             'name')
# 停留表
stay = pd.read_sql("SELECT * FROM dws_stay",
               con,
               parse_dates=['start_time', "end_time"])

# 带左边的信令
sig = pd.read_sql("""
    SELECT s.imsi, s.ts, c.lng,
    c.lat FROM dwd_signal s JOIN dim_cell c ON s.lac_cell = c.lac_cell
    """,
          con,
          parse_dates=["ts"])
n_users = sig['imsi'].nunique()

In [ ]:
# 查看结果
print(f"停留 {len(stay):,} 个；带坐标的信令 {len(sig):,} 条；用户 {n_users:,} 人")

## 把需求变成口径：谁说西塔傍晚的访客
甲方要的是：国庆傍晚到访西塔的人，这句话代码执行不了，先把它拆成需求

- 西塔停留:
停留质心离西塔中心不超过半径、持续不少于最短时长->hotspot.radius_m 500 米；stay.min_minutes 30 分钟
- 傍晚访客:
有西塔停留，而且这次停留的结束时刻不早于 17:00	visitor.evening_hour 17

In [ ]:
# 西塔停留：dws_stay 里每个停留已经算好了 dist_xita_m
xita = stay[stay['dist_xita_m'] <= R]

# 傍晚访客：西塔停留的，结束时刻大于当天到17:00
enening_cut = DAY_START + pd.Timedelta(hours=P['visitor']['evening_hour'])
enening = xita[xita['end_time'] >= enening_cut]

# 一个人可能有好几个西塔停留
visitors = pd.Index(sorted(enening['imsi'].unique()),name='imsi')

print(f"有西塔停留的：{xita['imsi'].nunique():,} 人；其中傍晚访客：{len(visitors):,} 人，"
      f"占全部用户 {len(visitors) / n_users:.1%}")

## 我想知道半径300， 500， 1000米，停留时长30分钟，60分钟，120分钟都是什么样的

```markdown
ll --tree -L 2 --icons --no-git --ignore-glob "image" --no-permissions --no-user --no-time --no-filesize

 .
├──  configs
│   └──  settings.yaml
├──  data
│   ├──  signal.db
│   ├──  signal.zip
│   └──  signal_before_D05.db
├──  docs
│   ├──  AGENTS.md
│   ├──  主题使用说明.md
│   ├──  主题扩展指南.md
│   ├──  发现清单.md
│   ├──  图表规范.md
│   ├──  测试说明.md
│   └──  需求改写.md
├──  markdown
│   ├──  D03-空间维度分析.html
│   ├──  D03-空间维度分析.zip
│   ├──  D04-学生任务.docx
│   ├──  D04-学生任务.md
│   ├──  D04.md
│   ├──  DAY007-课件.zip
│   ├──  前两天内容整理.md
│   ├──  数据字典.md
│   ├──  环境准备.md
│   ├──  环境配置2.md
│   ├──  综合评估报告-主题系统与稽核封装.md
│   └──  项目根路径定位.md
├──  notebook
│   ├──  D02-信令数据洞察-2.ipynb
│   ├──  D03-空间维度分析.ipynb
│   ├──  D04-这份数据是谁.ipynb
│   ├──  D1-项目启动与数据接入-重构之后.ipynb
│   ├──  D1-项目启动与数据接入.ipynb
│   ├──  D2-信令数据验收.ipynb
│   ├──  D5-信令客流洞察-数据流水线与性能.ipynb
│   ├──  D6-信令数据洞察-驻留识别.ipynb
│   └──  D7-信令数据分析-西塔访客分析.ipynb
├── 󰣞 src
│   └──  signal_insight
│       ├──  test
│       │   ├──  demo.py
│       │   ├──  maps.py
│       │   └──  theme.py
│       ├──  viz
│       │   ├──  styles
│       │   │   ├──  academic.mplstyle
│       │   │   ├──  base.mplstyle
│       │   │   ├──  business.mplstyle
│       │   │   ├──  default.mplstyle
│       │   │   └──  lively.mplstyle
│       │   ├──  callouts.py
│       │   ├──  heatmatrix.py
│       │   ├──  layers.py
│       │   ├──  mapcolors.py
│       │   ├──  maps.py
│       │   └──  theme.py
│       ├──  config.py
│       ├──  db.py
│       ├──  dims.py
│       ├──  dwd.py
│       ├──  dws.py
│       ├──  geo.py
│       ├──  pipeline.py
│       └──  quality.py
├──  test
│   ├──  contract
│   │   ├──  conftest.py
│   │   ├──  ods_schema.py
│   │   ├──  test_db_contract.py
│   │   ├──  test_dims_contract.py
│   │   ├──  test_dwd_contract.py
│   │   ├──  test_pipeline_contract.py
│   │   ├──  test_quality_contract.py
│   │   └──  test_real_db_smoke.py
│   ├──  unit
│   │   ├──  test_config.py
│   │   ├──  test_db.py
│   │   ├──  test_dims.py
│   │   ├──  test_dwd.py
│   │   ├──  test_geo.py
│   │   ├──  test_pipeline.py
│   │   └──  test_quality.py
│   └──  conftest.py
├──  pyproject.toml
├── 󰂺 README.md
└──  signal.workspace
```

In [ ]:
d_sig = geo.haversine(sig['lat'], sig['lng'], hot['lat'], hot['lng'])

rows = []
for r in [300, 500, 1000]:
    rows.append(
        {'半径': f"{r}米", '口径':"任意一条信令",
         "人数": sig.loc[d_sig <= r, 'imsi'].nunique()
        })
    for t_min in [30, 60, 120]:
        n = stay[(stay['dist_xita_m'] <= r) &
                 (stay['duration_min'] >= t_min)]['imsi'].nunique()
        rows.append(
        {"半径":f"{r}米","口径": f"停留>={t_min}分钟","人数":n})
caliber = pd.DataFrame(rows)

In [ ]:
caliber.to_sql("ads_caliber_visitors",
            con, if_exists="replace", index=False)
COLS = ['任意一条信令', "停留>=30分钟","停留>=60分钟","停留>=120分钟"]
mat = caliber.pivot(
    index='半径',columns='口径',values='人数').loc[["300米","500米","1000米"],COLS]

In [ ]:
# ── 出图 D7_01：热力矩阵（复用 viz.matrix：格子里写数字、按数值上色，hot 给本项目采用的格子加框）──
fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
m = viz.matrix(mat,
               ax=ax,
               hot=[("500米", "停留>=30分钟")],
               row_title="离西塔多近",
               col_title="待多久",
               fontsize=12,
               rotate=0)
lo, hi = int(mat.values.min()), int(mat.values.max())  # 标题里的数从数据算，不手敲
title = f"同样叫“西塔到访人数”，口径不同，结果在 {lo:,} 到 {hi:,} 人之间，相差 {hi / lo:.1f} 倍"
viz.finish(
    fig, title,
    "2018-10-03 全天，按“离西塔多近”和“待多久”两个口径统计的到访人数；加框 = 本项目采用的口径；口径：dwd_signal、dws_stay",
    SOURCE, "D7_01_到访人数口径对比.png")
plt.show()

“西塔到访人数”随口径在 1,836 到 5,282 人之间，相差 2.9 倍；本报告统一采用“500 米内停留 30 分钟以上”，为 3,603 人（图 D7_01）。

## 几点在哪？每人每个小时一个位置，三个商圈的一天
作业：需求：客户最常问的东西： "几点，在什么地方，有多少人"，简单来说就是得知道每个小时，每个商业街停留的人数。需要注意不是问你这个地区在指定的时间段出现过多少人，而是停留了多少人, 并最终完成可视化

你要处理的表是停留表：停留表只记录待够30分钟的地方，信令又时有时无，所有先给每个人每个小时定一个位置
口径：

1. 这个小时里面，有停留覆盖30分钟以上的，用覆盖时间最长的那个停留位置
2. 没有这样的停留：用这个小时所有信令位置的中位数，不可以这样子
3. 这个小时一条信令都没有：这个人这个小时不可定位，不能硬补，所以每个小时的可定位人数不一样，算占比是分母用当小时的可定位人数

提示：
- 把停留的其实时刻，换算成当天到第几小时的小数： 17:30 -> 17.5
- 一个人同一个小时可能被两个停留各覆盖30分钟，比如9:00-9:30在A区，9:30-10:00在B区，规则是去覆盖最长的，如果两者一样，去重保留第一次出现的
- 没被停留覆盖的人 * 小时，用信令位置的中位数
- 第三步： 拼起来，补上区县，商圈，到西塔的距离
| zone       |    0 |    1 |    2 |    3 |    4 |    5 |    6 |    7 |    8 |    9 |   10 |   11 |   12 |   13 |   14 |   15 |   16 |   17 |   18 |   19 |   20 |   21 |   22 |   23 |
|:-----------|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|-----:|
| 西塔       |  614 |  660 |  644 |  616 |  575 |  576 |  547 |  525 |  508 |  493 |  470 |  479 |  451 |  446 |  443 |  458 |  500 |  932 | 1639 | 2037 | 1980 | 1781 | 1661 | 1432 |
| 太原街商圈 |  235 |  243 |  265 |  270 |  265 |  263 |  282 |  301 |  296 |  305 |  288 |  309 |  301 |  304 |  344 |  365 |  421 |  442 |  382 |  334 |  362 |  425 |  428 |  391 |
| 中街商圈   |   53 |   59 |   59 |   58 |   59 |   61 |   65 |   71 |   93 |  132 |  185 |  241 |  253 |  251 |  281 |  288 |  297 |  221 |  183 |  141 |  133 |  129 |   89 |   72 |
| 可定位人数 | 3046 | 3363 | 3507 | 3611 | 3666 | 3740 | 3891 | 4045 | 4177 | 4243 | 4285 | 4393 | 4407 | 4445 | 4526 | 4615 | 4783 | 4967 | 5039 | 5069 | 5043 | 4945 | 4766 | 4287 |


## 待多久：数据在23:59分截止，时长只是下线
需求：傍晚访客在西塔待了多久？先给每个傍晚访客做一行：几点到、一共待多久、23 点后还在不在。

In [ ]:
gx = xita[xita['imsi'].isin(visitors)].groupby("imsi")

vis = pd.DataFrame(index=visitors)
# 当天首次到达西塔是几点？
vis['arrive_hour'] = ((gx['start_time'].min() - DAY_START).dt.total_seconds() /
                      3600).round(2)
# 在西塔一共待了多长时间
vis['xita_stay_min'] = gx['duration_min'].sum().round(1)

# 23点之后还在西塔吗？
vis['after_23'] = gx['end_time'].max() >= DAY_START + pd.Timedelta(
    hours=P['visitor']['late_hour'])

med_h = vis['xita_stay_min'].median() / 60
print(f"傍晚访客 {len(vis):,} 人：在西塔停留中位 {med_h:.1f} 小时；23 点后仍在的 {vis['after_23'].mean():.0%}")
vis.head(3).rename(index=mask_id)                

::: WARNING 注意
arrive_hour 是小数小时：17.41 是 17 点 25 分左右（0.41 × 60），不是 17 点 41 分。布尔列求 mean() 就是 True 的占比。
:::

In [ ]:
# 把时间处理好
v = vis[np.floor(vis['arrive_hour']).between(17, 22)]
cens = v.groupby(np.floor(v['arrive_hour']).astype(int))['after_23'].mean()
cens_all = vis['after_23'].mean()
cens.rename_axis('首次到达小时').rename('23点后扔在的占比').reset_index().to_sql(
    "ads_xita_censoring", con, if_exists='replace', index=False
)

In [ ]:
# ── 出图 D7_03：柱形图 + 参考线（普通 matplotlib，配色用 viz 的语义色）──
fig, ax = plt.subplots(figsize=(10, 5.0), layout="constrained")
ax.axhline(cens_all, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=0)  # 参考线压在柱子下面
bars = ax.bar([f"{h} 点" for h in cens.index],
              cens.values,
              color=MAIN,
              width=0.6,
              zorder=2)
ax.bar_label(bars,
             labels=[f"{x:.0%}" for x in cens.values],
             padding=4,
             color=INK)
ax.text(len(cens) - 0.45,
        cens_all,
        f"全部傍晚访客 {cens_all:.0%}",
        va="bottom",
        ha="left",
        color=MUTED,
        fontsize=9.5)
ax.set_xlim(-0.6, len(cens) + 0.9)
ax.set_ylim(0, 1.08)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("首次到达西塔的时刻", color=MUTED)
title = f"{cens_all:.0%} 的傍晚访客 23 点后仍在西塔，越晚到的越多：在西塔停留中位至少 {med_h:.1f} 小时"
viz.finish(
    fig, title,
    "按首次到达时刻分组，23:00 以后仍有西塔停留的访客占比；虚线 = 全部傍晚访客；数据截止 23:59，之后何时离开无法观测；口径：dws_stay",
    SOURCE, "D7_03_停留时长删失.png")
plt.show()

41% 的傍晚访客到 23 点仍在西塔，且越晚到达的比例越高；由于数据截止于 23:59，这部分人的停留时长只是下限，傍晚访客在西塔的停留时长中位数至少为 2.7 小时（图 D7_03）。

## 从哪来？过夜地在哪？

- 这群里从哪里来？只有一天的数据，看不到家，能看到的是凌晨待的最久的地方，叫过夜地

In [ ]:
# 每个人一个过夜地
NP = P['night']
n0 = DAY_START + pd.Timedelta(hours = NP['start_hour'])
n1 = DAY_START + pd.Timedelta(hours = NP['end_hour'])

# 每个停留和 [00:00, 06:00) 重叠多少分钟——这次直接用 Timestamp 算：
#   clip(upper=n1)：结束时刻晚于 06:00 的按 06:00 算（= min(结束, 06:00)）
#   clip(lower=n0)：开始时刻早于 00:00 的按 00:00 算（= max(开始, 00:00)）
#   相减得到时间差；.dt.total_seconds() / 60 换成分钟；clip(lower=0) 把“完全不重叠”的负数变成 0
overlap = stay['end_time'].clip(upper=n1) - stay['start_time'].clip(lower=n0)
stay['night_min']  = (overlap.dt.total_seconds() / 60).clip(lower=0)

# 每人取重叠最长的一个，而且至少 60 分钟：
#   先筛掉不足 60 分钟的 → 按 night_min 从小到大排（并列时按 stay_seq，结果可复现）
#   → groupby("imsi").tail(1) 取每组最后一行 = 重叠最长的那个
night = (stay[stay['night_min'] >= NP['min_overlap_minutes']]
        .sort_values(['night_min', 'stay_seq'])
        .groupby('imsi').tail(1)
        .set_index('imsi').sort_index()
        )
# 改列名，让下游一看就知道是“过夜地的”经纬度和区县；只留需要的列
night = night.rename(columns={"lng": "night_lng", "lat": "night_lat", "district": "night_district",
                              "dist_xita_m": "night_dist_xita_m"})[
    ["night_lng", "night_lat", "night_district", "night_dist_xita_m", "night_min"]]
db.replace_table(con, night.reset_index(), 'dws_night')
print(f"能识别过夜地的：{len(night):,} 人，占全部用户 {len(night) / n_users:.1%}；"
      f"傍晚访客里能识别的占 {visitors.isin(night.index).mean():.0%}")
night.head(3).rename(index=mask_id)

In [ ]:
# 过夜地距离西塔有多远，分档+中位数
km = night['night_dist_xita_m'] / 1000
bands = pd.cut(km,
               bins=[0, 1.5, 5, 15, np.inf],
               right=False,
               labels=['1.5公里内', '1.5-5公里', '5-15公里', '15公里以上'])
band_share = bands.value_counts(normalize=True).sort_values()
band_share.rename('占比').rename_axis('过夜地离西塔').reset_index().to_sql(
    "ads_night_distance", con, if_exists='replace', index=False)

# 过夜地落在那些区县： 人数+占比
nd = night['night_district'].value_counts().rename('人数').rename_axis(
    '区县').reset_index()

nd['占比'] = nd['人数'] / nd['人数'].sum()
nd.to_sql("ds_night_district", con, if_exists="replace", index=False)

top_d = nd.set_index('区县')['占比']

In [ ]:
med_km = km.median()

In [ ]:
# ── 出图 D7_04：累计分布（ECDF），背景色带标出四个距离档 ──
fig, ax = plt.subplots(figsize=(10, 5.2), layout="constrained")
for (a, b), lab, c_ in zip([(0, 1.5), (1.5, 5), (5, 15), (15, 30)], band_share.index,
                           [MAIN_LIGHT, "#EEF3F8", "#F6F8FA", "white"]):
    ax.axvspan(a, b, color=c_, lw=0, zorder=0)
    ax.text((a + b) / 2, 1.05, f"{lab}\n{band_share[lab]:.0%}", ha="center", fontsize=9, color=MUTED)
ax.ecdf(km, color=MAIN, lw=2.2)
ax.plot([med_km, med_km], [0, 0.5], color=INK, lw=1, ls="--"); ax.plot(med_km, 0.5, "o", color=MAIN, ms=6)
ax.text(med_km + 0.4, 0.52, f"中位数 {med_km:.1f} 公里", fontsize=10, color=INK, fontweight="bold")
ax.set_xlim(0, 30); ax.set_ylim(0, 1.16); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("过夜地到西塔中心的直线距离（公里）"); ax.set_ylabel("距离不超过横轴数值的人"); ax.grid(axis="both")
title = f"一半的人过夜地离西塔不到 {med_km:.1f} 公里，{band_share.iloc[-1]:.0%} 在 15 公里以外"
viz.finish(fig, title,
           f"能识别过夜地的 {len(night):,} 人（占全部用户 {len(night) / n_users:.0%}）；曲线上一点 = 距离不超过横轴数值的人数占比；直线距离，不是路程；口径：dws_night",
           SOURCE, "D7_04_过夜地距离.png")
plt.show()

能识别过夜地的样本用户（占 64%）中，一半的过夜地距西塔不到 2.9 公里，6% 在 15 公里以外（图 D7_04）。

In [ ]:
# ── 出图 D7_05：六边形热力地图（复用 Regions.base / hexbin / zoom）+ 区县条形图（掌握：讲师演示）──
city = viz.maps.Regions(DISTRICTS)
K = P["privacy"]["min_cell_users"]
fig, (a1, a2) = plt.subplots(1,
                             2,
                             figsize=(12, 6.0),
                             layout="constrained",
                             width_ratios=[2, 1])
city.base(a1, scale=False, north=False)  # 底色 + 区界
hb = city.hexbin(a1,
                 night,
                 size_m=800,
                 lng="night_lng",
                 lat="night_lat",
                 how="count",
                 min_value=K)  # 少于 5 人的格子不画
ins = a1.inset_axes([0.0, 0.66, 0.26, 0.34])  # 左上角小图：全市，标出放大的范围
city.base(ins, scale=False, north=False, borders=False)
viz.maps.zoom(ins, a1, hot["lng"], hot["lat"],
              (10000, 8000))  # 主图放大到西塔周围 20 × 16 公里
viz.maps.circle(a1, hot["lng"], hot["lat"], R, ec=INK)  # 西塔 500 米
viz.maps.scale_bar(a1, 2, x=0.05, y=0.04)
viz.maps.north_arrow(a1)
AT = {
    "和平区": (123.39, 41.765),
    "沈河区": (123.50, 41.79),
    "大东区": (123.49, 41.855),
    "皇姑区": (123.43, 41.86),
    "铁西区": (123.32, 41.79),
    "于洪区": (123.305, 41.825),
    "浑南区": (123.47, 41.745)
}
city.text(a1,
          dict.fromkeys(AT, np.nan),
          at=AT,
          fs=9,
          nameprops=dict(color=MUTED))
viz.maps.colorbar(hb, a1, label="六边形内的过夜人数（人，格宽 800 米，对数色阶）")
shown = hb.cells.loc[~hb.cells["suppressed"].astype(bool),
                     "value"].sum() / len(night)

d7 = top_d.head(7)[::-1]
bars = a2.barh(d7.index,
               d7.values,
               color=[MAIN if k == "和平区" else GRAY for k in d7.index],
               height=0.62)
a2.bar_label(bars,
             labels=[f"{v:.0%}" for v in d7.values],
             padding=3,
             fontsize=9.5,
             color=INK)
a2.set_xlim(0, d7.max() * 1.3)
a2.set_xticks([])
a2.grid(False)
a2.tick_params(axis="y", labelcolor=INK)
a2.set_title("过夜地所在区县（前 7）",
             loc="left",
             fontsize=10.5,
             color=INK,
             fontweight="bold")
title = f"过夜地集中在西塔周边，和平区占 {top_d['和平区']:.0%}"
viz.finish(
    fig, title,
    f"左：主城区放大图，每个六边形内的过夜人数，少于 {K} 人的格子不显示（显示的格子覆盖 {shown:.0%} 的人）；虚线圈 = 西塔 500 米；\n"
    f"过夜地 = 与 0–6 点重叠最长且 ≥ 60 分钟的停留；共 {len(night):,} 人；口径：dws_night", SOURCE,
    "D7_05_过夜地分布.png")
plt.show()

能识别过夜地的样本用户中，过夜地集中在西塔周边，和平区占 38%，铁西区、皇姑区各占 15%（图 D7_05）。

## 放课表入库,发现表写进ads_findings

In [ ]:
# 傍晚访客表加上过夜地两列。join 按索引（imsi）对齐：没有过夜地的人这两列是空值（NaN），不会被丢掉
vis = vis.join(night[["night_district", "night_dist_xita_m"]])

db.replace_table(con, vis.reset_index(), "dws_visitor")  # 明天 D08 只读它、不改它
print(
    f"dws_visitor：{len(vis):,} 行；其中能识别过夜地的 {vis['night_district'].notna().sum():,} 人"
)

In [ ]:
# 今天的发现：句子里的每个数字都来自上面的变量，不手敲
findings = pd.DataFrame([
    ("F22",
     f"国庆当天在西塔 500 米内停留 30 分钟以上的有 {xita['imsi'].nunique():,} 人，其中 {len(visitors):,} 人到 17 点后仍在（傍晚访客）",
     "dws_stay", "西塔停留 = 质心距西塔 ≤ 500 米且 ≥ 30 分钟；傍晚访客 = 该停留结束时刻 ≥ 17:00"),
    ("F23",
     f"“西塔到访人数”随口径在 {lo:,} 到 {hi:,} 人之间，相差 {hi / lo:.1f} 倍；本项目统一采用 500 米 / 30 分钟口径，为 {mat.loc['500米', '停留>=30分钟']:,} 人",
     "图 D7_01", "半径 300 / 500 / 1000 米 × 任意信令或停留 30 / 60 / 120 分钟"),
    ("F25",
     f"{cens_all:.0%} 的傍晚访客 23 点后仍在西塔，越晚到的比例越高；在西塔停留中位至少 {med_h:.1f} 小时",
     "图 D7_03", "数据截止 23:59，23 点后仍在者的停留时长是下限（右删失）"),
    ("F26",
     f"{len(night) / n_users:.0%} 的用户能识别过夜地；其中一半离西塔不到 {med_km:.1f} 公里，{band_share.iloc[-1]:.0%} 在 15 公里以外，和平区占 {top_d['和平区']:.0%}",
     "图 D7_04、D7_05", "过夜地 = 与 0–6 点重叠最长且 ≥ 60 分钟的停留（不等于家）；直线距离"),
],
                        columns=["编号", "发现", "证据", "口径"])

# write_partition 是 D05 写的“按分区先删后写”：这里按“来源”分区，D07 重跑多少次都只有一份 D07 的发现
db.write_partition(con,
                   findings.assign(来源="D07"),
                   "ads_findings",
                   "D07",
                   dt_col="来源")
pd.read_sql("SELECT 编号, 发现 FROM ads_findings WHERE 来源 = 'D07'", con)

In [ ]:
# ── 出图 D7_05：三栏 ── 左：全市（轮廓+区名，框出放大范围）｜中：西塔周边放大（连线指向）｜右：区县人数 ──
ZOOM_C = "#D9822B"                  # 放大框、连线、放大图边框同一个橙色：读者顺着颜色就能从左图找到中图
HALF = (10000, 8000)                # 放大范围：半宽 10 km × 半高 8 km（20 × 16 公里）
SIZE_M = 800                        # 左、中两张图同一格宽、同一色阶上限，共用一根色条
BLACK = "#000000"
LINE = dict(color="#7B8696", lw=0.8)            # 区界：比默认的白线深，叠在热力上也看得清

city = viz.maps.Regions(DISTRICTS)
K = P["privacy"]["min_cell_users"]
hx = dict(lng="night_lng", lat="night_lat", how="count", min_value=K)

fig, (a1, a2, a3) = plt.subplots(
    1, 3, figsize=(17, 6.8), layout="constrained",
    width_ratios=[city.aspect * 1.3, HALF[0] / HALF[1], 0.95])   # 左栏多留 30%：放引线标注的区名

# ── 左：全市 ───────────────────────────────────────────────
city.base(a1, borders=LINE, outline=dict(color=INK, lw=1.5), north=dict(x=0.95, y=0.95))
hb = city.hexbin(a1, night, size_m=SIZE_M, **hx)
city.edges(a1, ["和平区"], color=MAIN_DARK, lw=1.8)         # 和平区加粗描边：对应右栏高亮的柱子
city.label(a1, dict.fromkeys(city.names, np.nan),
           inside=False, fs=9.5,split="balance",
           nameprops=dict(color=BLACK))                      # 放得下的写在区里，放不下的引到图外
a1.set_title("全市过夜地分布", loc="left", fontsize=10.5, color=INK, fontweight="bold")

# ── 中：西塔周边放大 ───────────────────────────────────────
city.base(a2, scale=False, north=False, fill=False, borders=LINE, outline=dict(color=INK, lw=1.5))
viz.maps.zoom(a1, a2, hot["lng"], hot["lat"], HALF,
              label="放大区", color=ZOOM_C, connect=True)    # connect=True：左图的框两角连线到中图两角
city.hexbin(a2, night, size_m=SIZE_M, vmax=hb.norm.vmax, **hx)
city.edges(a2, ["和平区"], color=MAIN_DARK, lw=1.8)
viz.maps.circle(a2, hot["lng"], hot["lat"], R, ec="red")      # 西塔 500 米
ry = viz.maps.m2deg(R, hot["lat"])[1]
a2.text(hot["lng"], hot["lat"] + ry * 1.15, f"西塔 {R:,.0f} 米", ha="center", va="bottom",
        fontsize=10.5, color=BLACK, fontweight="bold", path_effects=viz.maps.HALO, zorder=7)
viz.maps.scale_bar(a2, 2, x=0.05, y=0.04)
AT = {"和平区": (123.39, 41.765), "沈河区": (123.50, 41.79), "大东区": (123.49, 41.855),
      "皇姑区": (123.43, 41.86), "铁西区": (123.32, 41.79), "于洪区": (123.305, 41.825),
      "浑南区": (123.47, 41.745)}
city.text(a2, dict.fromkeys(AT, np.nan), at=AT, fs=10.5, halo=True,
          nameprops=dict(color=BLACK, fontweight="bold"))   # 区名：黑色粗体 + 白边，压在热力上也清楚
a2.set_title("放大：西塔周边 20 × 16 公里", loc="left", fontsize=10.5, color=ZOOM_C, fontweight="bold")

viz.maps.colorbar(hb, [a1, a2], label=f"六边形内的过夜人数（人，格宽 {SIZE_M} 米，对数色阶）")
shown = hb.cells.loc[~hb.cells["suppressed"].astype(bool), "value"].sum() / len(night)

# ── 右：区县人数（前 7 + 其余合并，合起来是 100%）───────────
d = nd.sort_values("人数", ascending=False).reset_index(drop=True)
head, rest = d.head(7), d.iloc[7:]
bars_df = pd.concat([head, pd.DataFrame({"区县": [f"其余 {len(rest)} 项（含市外）"],
                                          "人数": [rest["人数"].sum()],
                                          "占比": [rest["占比"].sum()]})], ignore_index=True)[::-1]
colors = [MAIN if k == "和平区" else (FAINT if k.startswith("其余") else GRAY) for k in bars_df["区县"]]
bars = a3.barh(bars_df["区县"], bars_df["人数"], color=colors, height=0.62)
a3.bar_label(bars, labels=[f"{n:,} 人 · {p:.0%}" for n, p in zip(bars_df["人数"], bars_df["占比"])],
             padding=4, fontsize=9.5, color=INK)
a3.set_xlim(0, bars_df["人数"].max() * 1.5)
a3.set_xticks([]); a3.grid(False)
a3.tick_params(axis="y", labelcolor=INK)
a3.set_title("过夜地所在区县", loc="left", fontsize=10.5, color=INK, fontweight="bold")
top3 = head["占比"].head(3).sum()
a3.text(0.0, -0.04, f"前三区合计 {top3:.0%}", transform=a3.transAxes,
        ha="left", va="top", fontsize=9, color=MUTED)

title = f"过夜地集中在西塔周边，和平区占 {d.loc[d['区县'] == '和平区', '占比'].iloc[0]:.0%}"
viz.finish(
    fig, title,
    f"左：全市；中：放大到西塔周边（橙框与连线 = 放大范围）；虚线圈 = 西塔 {R:,.0f} 米；少于 {K} 人的格子不显示（显示的格子覆盖 {shown:.0%} 的人）。\n"
    f"过夜地 = 与 0–6 点重叠最长且 ≥ 60 分钟的停留；共 {len(night):,} 人；口径：dws_night", SOURCE,
    "D7_05_过夜地分布.png")
plt.show()